# Signalement des anomalies d'adresses email — EJ FINESS

## 1. Imports et connexion

In [1]:
%run ../../config/config.ipynb

import sys, os
sys.path.insert(0, os.path.abspath('../..'))

import warnings
warnings.filterwarnings("ignore")

import pandas as pd
from pathlib import Path

from src.email_checker import (
    analyser_emails, marquer_doublons, evaluer_correspondance,
    charger_tlds_iana, charger_bases_noms,
)

pd.set_option("display.max_colwidth", 80)
print("Imports OK")

Looking in indexes: http://10.4.3.215:8081/repository/pypi-all/simple
Note: you may need to restart the kernel to use updated packages.
Connexion OK
Imports OK


## 2. Référentiels externes — TLD IANA + bases de noms (filtrées par fréquence) + base communes france

In [2]:
DOSSIER_REF = Path("../../data/referentiels")

charger_tlds_iana(cache=DOSSIER_REF / "tlds-alpha-by-domain.txt")

df_prenoms    = pd.read_csv(DOSSIER_REF / "prenom.csv",     sep=",", dtype=str)
df_patronymes = pd.read_csv(DOSSIER_REF / "patronymes.csv", sep=",", dtype=str)
charger_bases_noms(df_prenoms, df_patronymes)

from src.email_checker import PRENOMS, PATRONYMES, TLDS_VALIDES, SEUIL_PRENOM, SEUIL_PATRONYME
print(f"TLD valides                      : {len(TLDS_VALIDES):,}")
print(f"Prénoms retenus (sum >= {SEUIL_PRENOM})     : {len(PRENOMS):,}")
print(f"Patronymes retenus (count >= {SEUIL_PATRONYME}) : {len(PATRONYMES):,}")

TLD valides                      : 1,437
Prénoms retenus (sum >= 500)     : 1,309
Patronymes retenus (count >= 100) : 14,710


In [3]:
df_communes = pd.read_csv(DOSSIER_REF / "communes-france-2026.csv", dtype=str)

from src.email_checker import charger_geo_insee
stats_geo = charger_geo_insee(df_communes, seuil_population=2000)
print(f"Villes retenues (pop >= 2000) : {stats_geo['villes']:,}")
print(f"Départements : {stats_geo['departements']} · Régions : {stats_geo['regions']}")

Villes retenues (pop >= 2000) : 5,200
Départements : 101 · Régions : 18


## 3. Chargement EJ (BIcoeur) + jointure avec le CSV des emails

In [4]:
query_ej = """
    SELECT
        idstructure_stru,
        nmfinessej_stru,
        raisonsociale_stru,
        cdcommune_stru,
        lbvoie_stru
    FROM BICOEUR_DWH_SNAPSHOT.dbo.dwh_structure
    WHERE topsource_stru = 'FINESS'
      AND typeidpm_stru  = 'EJ'
      AND (dtfermestruct_stru IS NULL OR dtfermestruct_stru >= SYSDATETIME())
"""
df_ej_base = pd.read_sql(query_ej, conn)
print(f"EJ FINESS actifs (BIcoeur) : {len(df_ej_base):,}")

FICHIER_EMAILS_EJ = Path("../../data/Export_email_EJ_FinessProd_20260615.csv")  # adapte le nom

def lire_csv(chemin):
    for enc in ("utf-8", "latin-1"):
        for sep in (";", ",", "\t", "|"):
            try:
                d = pd.read_csv(chemin, sep=sep, dtype=str, encoding=enc)
                if d.shape[1] >= 2:
                    print(f"CSV lu (sep='{sep}', encoding='{enc}')")
                    return d
            except Exception:
                continue
    raise ValueError("Impossible de lire le CSV avec les séparateurs testés")

df_emails = lire_csv(FICHIER_EMAILS_EJ)
df_emails = df_emails.rename(columns={
    df_emails.columns[0]: "nmfinessej_stru",
    df_emails.columns[1]: "email_stru",
})[["nmfinessej_stru", "email_stru"]].copy()

df_emails["nmfinessej_stru"] = df_emails["nmfinessej_stru"].astype(str).str.strip()
df_emails["email_stru"]      = df_emails["email_stru"].astype(str).str.strip()
df_emails = df_emails[~df_emails["email_stru"].str.lower().isin(["nan", "none", "null", ""])]
df_emails = df_emails.drop_duplicates(subset=["nmfinessej_stru"], keep="first")
print(f"FINESS avec email non vide : {len(df_emails):,}")

df_ej_base["nmfinessej_stru"] = df_ej_base["nmfinessej_stru"].astype(str).str.strip()
long_max = max(df_ej_base["nmfinessej_stru"].str.len().max(),
               df_emails["nmfinessej_stru"].str.len().max())
df_ej_base["nmfinessej_stru"] = df_ej_base["nmfinessej_stru"].str.zfill(long_max)
df_emails["nmfinessej_stru"]  = df_emails["nmfinessej_stru"].str.zfill(long_max)

df_ej = df_ej_base.merge(df_emails, on="nmfinessej_stru", how="left")

df_ej["email_stru"] = df_ej["email_stru"].astype("string").str.strip()
df_ej["email_stru"] = df_ej["email_stru"].mask(
    df_ej["email_stru"].isna() | df_ej["email_stru"].str.lower().isin(["nan", "none", "null"]),
    ""
)
df_ej["email_stru"] = df_ej["email_stru"].astype(str)

df_ej["departement"] = df_ej["cdcommune_stru"].astype(str).str[:2]

n_avec = (df_ej["email_stru"] != "").sum()
print(f"\nAprès jointure :")
print(f"  EJ total      : {len(df_ej):,}")
print(f"  EJ avec email : {n_avec:,}  ({n_avec/len(df_ej)*100:.1f} %)")
print(f"  EJ sans email : {len(df_ej) - n_avec:,}")

EJ FINESS actifs (BIcoeur) : 54,185
CSV lu (sep=';', encoding='utf-8')
FINESS avec email non vide : 34,293

Après jointure :
  EJ total      : 54,185
  EJ avec email : 29,130  (53.8 %)
  EJ sans email : 25,055


## 4. Détection des anomalies

In [5]:
print("Analyse des anomalies + classification...")
df = analyser_emails(df_ej, col_email="email_stru")
df = marquer_doublons(df, col_email_norm="email_norm", col_id="nmfinessej_stru")
print(f"Analyse terminée — {len(df):,} EJ traités.")

Analyse des anomalies + classification...
Analyse terminée — 54,185 EJ traités.


## 5. Correspondance partie locale ↔ raison sociale / adresse

In [6]:
def _corresp_ej(row):
    return evaluer_correspondance(
        local_part=row["local_part"],
        raison_principale=row.get("raisonsociale_stru", ""),
        raison_parent="",
        adresse=row.get("lbvoie_stru", ""),
        libelle_principal="(EJ)",
    )

df["correspondance"] = ""
masque = df["local_part"] != ""
df.loc[masque, "correspondance"] = df.loc[masque].apply(_corresp_ej, axis=1)

print("Répartition des correspondances :")
print(df[df["correspondance"] != ""]["correspondance"].value_counts().to_string())

Répartition des correspondances :
correspondance
Aucune correspondance            15637
Raison sociale (EJ)              12542
Raison sociale (EJ) + Adresse      761
Adresse                            190


In [ ]:
from src.email_checker import detecter_geo

geo = df.apply(lambda row: detecter_geo(row["local_part"], row.get("cdcommune_stru")), axis=1)
df["geo_local"]      = geo.apply(lambda x: x["geo_local"])
df["geo_concordant"] = geo.apply(lambda x: x["geo_concordant"])

apercu = df[(df["geo_local"] != "") & (df["geo_concordant"] == False)]
print(f"Emails portant un lieu : {(df['geo_local'] != '').sum():,}")
print(f"  dont concordants     : {(df['geo_concordant'] == True).sum():,}")
print(f"  dont NON concordants : {len(apercu):,}")
apercu[["raisonsociale_stru", "email_stru", "geo_local"]].head(10)

## 6. Distribution + classification

In [8]:
dist_niveau = df["niveau_anomalie"].value_counts().sort_index()
labels = {0: "Valides", 1: "Critiques (niveau 1)", 2: "Qualité (niveau 2)"}

print("─" * 55)
print("DISTRIBUTION GLOBALE DES SIGNALEMENTS")
print("─" * 55)
for niveau, n in dist_niveau.items():
    print(f"  {labels.get(niveau, f'Niveau {niveau}'):<30} : {n:>8,}  ({n/len(df)*100:5.1f} %)")
print("─" * 55)

print("\nDÉTAIL PAR TYPE D'ANOMALIE")
print("─" * 55)
detail = (
    df[df["niveau_anomalie"] > 0]
    .groupby(["niveau_anomalie", "code_anomalie"])
    .size().reset_index(name="nb")
    .sort_values(["niveau_anomalie", "nb"], ascending=[True, False])
)
for _, r in detail.iterrows():
    print(f"  [{r['niveau_anomalie']}] {r['code_anomalie']:<22} : {r['nb']:>6,}")

classif = df[df["type_local"] != ""]
if len(classif) > 0:
    print("\nClassification type_local :")
    print(classif["type_local"].value_counts().to_string())

───────────────────────────────────────────────────────
DISTRIBUTION GLOBALE DES SIGNALEMENTS
───────────────────────────────────────────────────────
  Valides                        :   12,323  ( 22.7 %)
  Critiques (niveau 1)           :   25,134  ( 46.4 %)
  Qualité (niveau 2)             :   16,728  ( 30.9 %)
───────────────────────────────────────────────────────

DÉTAIL PAR TYPE D'ANOMALIE
───────────────────────────────────────────────────────
  [1] EMAIL_VIDE             : 25,055
  [1] TLD_INEXISTANT         :     21
  [1] DOMAINE_SANS_POINT     :     20
  [1] EXTENSION_VIDE         :     19
  [1] DOMAINE_INEXISTANT     :     15
  [1] CARACTERE_INTERDIT     :      1
  [1] DOMAINE_VIDE           :      1
  [1] ESPACE                 :      1
  [1] EXTENSION_TYPO         :      1
  [2] EMAIL_GRAND_PUBLIC     : 15,455
  [2] DOUBLON                :  1,273

Classification type_local :
type_local
INDETERMINE          10245
INSTITUTIONNEL        7864
GENERIQUE             7246
NOMINA

## 7. Export Excel — Rapport de signalement

In [9]:
from openpyxl.styles import PatternFill, Font, Alignment

DOSSIER_SORTIE = Path("../../results/email")
DOSSIER_SORTIE.mkdir(parents=True, exist_ok=True)
FICHIER_SORTIE = DOSSIER_SORTIE / "signalement_emails_ej.xlsx"

COLS_EXPORT = [
    "nmfinessej_stru", "raisonsociale_stru",
    "cdcommune_stru", "departement", "lbvoie_stru",
    "email_stru", "email_norm",
    "code_anomalie", "libelle_anomalie", "suggestion",
    "type_local", "type_domaine", "correspondance",
    "geo_local", "geo_concordant",
]
cols_dispo = [c for c in COLS_EXPORT if c in df.columns]

def style_entete(ws, couleur):
    for cell in ws[1]:
        cell.fill = PatternFill("solid", start_color=couleur, end_color=couleur)
        cell.font = Font(bold=True, color="FFFFFF", name="Arial", size=10)
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
    ws.freeze_panes = "A2"
    ws.row_dimensions[1].height = 28

def remplir(ws, couleur):
    for row in ws.iter_rows(min_row=2):
        for cell in row:
            cell.fill = PatternFill("solid", start_color=couleur, end_color=couleur)
            cell.font = Font(name="Arial", size=9)

def auto_width(ws, max_w=50):
    for col in ws.columns:
        w = max((len(str(c.value or "")) for c in col), default=10)
        ws.column_dimensions[col[0].column_letter].width = min(w + 3, max_w)

n_total    = len(df)
n_vides    = (df["code_anomalie"] == "EMAIL_VIDE").sum()
n_renseign = max(n_total - n_vides, 1)

df_vides     = df[df["code_anomalie"] == "EMAIL_VIDE"][cols_dispo]
df_critiques = df[(df["niveau_anomalie"] == 1) & (df["code_anomalie"] != "EMAIL_VIDE")][cols_dispo]
df_doublons  = df[df["code_anomalie"] == "DOUBLON"][cols_dispo]
df_grandpub  = df[df["code_anomalie"] == "EMAIL_GRAND_PUBLIC"][cols_dispo]
df_valides   = df[df["niveau_anomalie"] == 0][cols_dispo]

# --- ajout : vue croisée par type de partie locale, tous type_domaine confondus (propre/public/doublon) ---
df_nominatifs   = df[df["type_local"].isin(["NOMINATIF", "NOMINATIF_PARTIEL"])][cols_dispo]
df_indetermines = df[df["type_local"] == "INDETERMINE"][cols_dispo]

df_synth = pd.DataFrame([
    {"Catégorie": "EJ total",                "Nombre": n_total,           "Part": "100 %"},
    {"Catégorie": "Email absent (vide)",      "Nombre": n_vides,           "Part": f"{n_vides/n_total*100:.1f} %"},
    {"Catégorie": "—",                        "Nombre": "",                "Part": ""},
    {"Catégorie": "[1] Anomalies critiques",  "Nombre": len(df_critiques), "Part": f"{len(df_critiques)/n_renseign*100:.1f} %"},
    {"Catégorie": "[2] Doublons",             "Nombre": len(df_doublons),  "Part": f"{len(df_doublons)/n_renseign*100:.1f} %"},
    {"Catégorie": "[2] Grand public",         "Nombre": len(df_grandpub),  "Part": f"{len(df_grandpub)/n_renseign*100:.1f} %"},
    {"Catégorie": "[0] Emails valides",       "Nombre": len(df_valides),   "Part": f"{len(df_valides)/n_renseign*100:.1f} %"},
    {"Catégorie": "—",                           "Nombre": "",                "Part": ""},
    {"Catégorie": "Vue croisée (hors total ci-dessus, quel que soit le domaine)", "Nombre": "", "Part": ""},
    {"Catégorie": "Nominatifs (NOMINATIF + PARTIEL)", "Nombre": len(df_nominatifs),   "Part": f"{len(df_nominatifs)/n_renseign*100:.1f} %"},
    {"Catégorie": "Indéterminés",                     "Nombre": len(df_indetermines), "Part": f"{len(df_indetermines)/n_renseign*100:.1f} %"},
])

with pd.ExcelWriter(FICHIER_SORTIE, engine="openpyxl") as writer:
    df_synth.to_excel(writer, sheet_name="Synthèse", index=False)
    style_entete(writer.sheets["Synthèse"], "1F3864")
    auto_width(writer.sheets["Synthèse"], max_w=40)

    for nom, donnees, ent, fond in [
        ("Emails_Vides",        df_vides,     "C62828", "F8CECC"),
        ("Anomalies_Critiques", df_critiques, "C62828", "F8CECC"),
        ("Doublons",            df_doublons,  "F57C00", "FFF2CC"),
        ("Grand_Public",        df_grandpub,  "F57C00", "FFF2CC"),
        ("Emails_Valides",      df_valides,   "2E7D32", "EBF5EB"),
        ("Nominatifs",          df_nominatifs,   "1565C0", "D6E4F0"), 
        ("Indetermines",        df_indetermines, "6A1B9A", "E8DAEF"),
    ]:
        if len(donnees) > 0:
            donnees.to_excel(writer, sheet_name=nom, index=False)
            style_entete(writer.sheets[nom], ent)
            remplir(writer.sheets[nom], fond)
            auto_width(writer.sheets[nom])

print(f"Export EJ OK → {FICHIER_SORTIE.resolve()}")
print(f"\nFeuilles produites :")
print(f"  Emails_Vides         : {len(df_vides):,}")
print(f"  Anomalies_Critiques  : {len(df_critiques):,}")
print(f"  Doublons             : {len(df_doublons):,}")
print(f"  Grand_Public         : {len(df_grandpub):,}")
print(f"  Emails_Valides       : {len(df_valides):,}")
print(f"  Nominatifs           : {len(df_nominatifs):,}")
print(f"  Indetermines         : {len(df_indetermines):,}")

Export EJ OK → /home/jovyan/work/signalement_data_contact/results/email/signalement_emails_ej.xlsx

Feuilles produites :
  Emails_Vides         : 25,055
  Anomalies_Critiques  : 79
  Doublons             : 1,273
  Grand_Public         : 15,455
  Emails_Valides       : 12,323
  Nominatifs           : 3,696
  Indetermines         : 10,245
